# Industrial Cable Defect Detection using Autoencoders
######

## Project Report
#### 50.039 Theory and Practice of Deep Learning (2026)
###### Deadline: 18 April 2026, 11:59 PM

####

#### Group 7
- Ang Li En Eldrick (1006908)
- Malvin Ken Sudirgo (1007164)
- Toh Jia Le (1007004)

######
**GitHub Repository:** https://github.com/jialetoh/50039-proj-group07-2026

---

## 1. Project Topic

### 1.1 Problem
In industrial manufacturing, manual inspection of cables for structural integrity is prone to human error and fatigue. Inspection mistakes can lead to system instability and sensor failure, potentially causing injuries and even death.

### 1.2 Objective
This project aims to do **pixel-level anomaly detection** and **binary segmentation** of cable defects, to potentially automate cable inspections using deep learning. Using an **unsupervised approach**, models will be trained on normal cable images only. Then, they will be evaluated based on how well they can detect and identify anomalous regions in defective cable images.

---

## 2. Dataset

### 2.1 Dataset Overview

This project uses the **cable** category from the **MVTec Anomaly Detection (MVTec AD)** dataset.

- Dataset obtained from Kaggle (https://www.kaggle.com/datasets/ipythonx/mvtec-ad?select=cable)
- More information available at the official MVTec website (https://www.mvtec.com/company/research/datasets/mvtec-ad).
- More detailed information about the dataset can be found in `notebooks/01_data_exploration.ipynb`.

### 2.2 Dataset Composition

The cable category contains a total of **374 samples**:

- 282 are **normal** samples of defect-free cables.
- 92 are **anomalous** samples of cables containing visible defects.

All samples consist of one **3-channel** (RGB) PNG image of a cable, with a resolution of 1024×1024 pixels. Additionally, every anomalous sample has a corresponding **ground-truth segmentation mask** showing exactly where the defects are in the cable image. Each mask is a 1-channel grayscale PNG image at the same resolution (1024×1024), stored in the `data/cable/ground_truth/` directory.

### 2.3 Defect Categories

The 92 anomalous samples are divided into **8 different types of defects**, with 10 to 14 samples per defect category:

1. bent_wire
2. cable_swap
3. cut_inner_insulation
4. cut_outer_insulation
5. missing_cable
6. missing_wire
7. poke_insulation
8. combined

Figure 1 below provides a visual overview of the dataset. Three example images are shown for each defect category (and normal cables are labelled as "good").

<div style="text-align: center;">
    <img src="../figures/Fig1_dataset_overview.png" style="display: block; margin: 0 auto; max-width: 80%;">
    <br>
    <i>Figure 1. Overview of normal and anomalous cable samples across all eight defect categories.</i>
    <br>
</div>

Figure 2 below shows anomalous samples from each category and how their corresponding ground-truth segmentation mask highlights defective regions in the image.

<div style="text-align: center;">
    <img src="../figures/Fig2_segmask_preview.png" style="display: block; margin: 0 auto; max-width: 80%;">
    <br>
    <i>Figure 2. Anomalous cable samples from each defect category with their corresponding ground-truth segmentation masks.</i>
    <br>
</div>

### 2.4 Original Dataset Split

# **====== continue editing here. above this section is already done =======**

The dataset provided is already split into two sets:

- Train: 224 normal samples
- Test: 58 normal + 92 anomalous samples

This follows a standard Only normal images are used for training in an unsupervised setup, so a model only learns what is "normal", allowing it to flag anomalous images.



### 2.5 Relevance to Project Theme

#### Class Imbalance
The anomalous samples make up ~25% of the dataset,

#### Real-World Relevance

The Real-World Constraint: You can explain that MVTec AD is explicitly built to mimic industrial optical inspection tasks. The defects (cuts, missing components, pokes) are not synthetic noise or artificially generated digital artifacts; they are physical, real-world manufacturing errors captured with high-resolution industrial sensors.

The Class Imbalance Constraint: The dataset perfectly models the natural extreme imbalance found in manufacturing. In the real world, defective products are rare, making it impossible to build a massive dataset of balanced normal/abnormal training data. Because the training set is completely devoid of defective samples (0 defects vs. 224 normal), your model is forced to learn the distribution of "normalcy" and flag deviations, perfectly satisfying the imbalance requirement.



The dataset is relatively small compared to typical deep learning datasets,

Dataset constraint fall under the category of anomaly detection.
This means your dataset must exhibit class imbalance, where the number of “normal” samples is
significantly greater than the number of “abnormal” samples. Such imbalance is common in many
real-world applications and requires careful model design, training, and evaluation.

The dataset is similar to a real-world industrial inspection setting where the total amount of data is small, defects  are rare

Although small, this distribution is representative of industrial inspection settings, where normal samples are relatively abundant, defects are rare, and abnormal patterns may vary across different defect types.

there are only 10+ samples for each defect, which we think is similar to a real-world setting, where there are many types of defects, and the defects can even overlap/combine, but defects are rare and normal samples are abundant in a real-world scenario, but overall, data may still be hard to obtain.

The cable subset is relatively small compared to many deep learning datasets, which often contain thousands of training images. This makes the task more challenging, especially at the pixel level. In addition, the limited number of anomalous samples per defect type makes it difficult to learn defect-specific patterns directly, which motivates the use of anomaly detection methods trained only on normal images.

The dataset is relatively small, consisting of only 282 “normal” samples and 92 “abnormal” samples have been further divided into 8 different defect types, showing class imbalance.

## 3. Approach

### 3.1 Dataset Split
- Training set: normal images only
- Validation set: 15% split from the normal training images
- Test set: official MVTec cable test set containing both normal and anomalous images

MVTec cable subset provides only training and test splits, so the original training set of normal images was further divided into training and validation subsets for model development.

- **Training set:** 224 normal images  
- **Validation set:** 58 normal images  
- **Test set:** 58 normal images and 92 anomalous images  

The validation set was used to monitor reconstruction performance during training, while the test set was used for final evaluation of anomaly detection and pixel-level defect segmentation.

### 3.2 Dataset Preprocessing
Describe:
- image resizing
- normalization
- preprocessing steps applied
- whether data augmentation was used

**Example:**
- Resize all images to `?`
- Normalize pixel values to `?`
- Baseline model trained without augmentation
- Later experiments include mild augmentation such as small rotation and translation

### 3.3 Inputs and Outputs
**Input:**  
- 1024x1024 RGB image (3-channel)

**Output:**  
- 1024x1024 binary segmentation mask, where a pixel value of 0 denotes a normal
background region and 1 denotes an anomalous/defective region.

### 3.3 Model Architecture

#### 3.3.1 Inspiration for Our Approach
why:
- unsupervised
- an autoencoder baseline for unsupervised anomaly detection
- Convolutional autoencoders for image reconstruction
- a pretrained encoder variant
- augmentation / tuning experiments
- comparison against stronger methods
- inspiration from State-of-the-art methods such as PatchCore, PaDiM, FastFlow, Reverse Distillation, etc.??

#### 3.3.2 Baseline Model
Describe the baseline convolutional autoencoder:
- encoder layers
- bottleneck
- decoder layers
- activation functions

Include:
- a simple architecture diagram
- code snippet (Brief explanation of how to recreate our model from scratch)
- explanation of why this architecture was chosen
- Model definitions are in `src/models.py`

#### 3.3.3 Improved Model
Describe the improved version using a pretrained encoder:
- encoder backbone used
- whether weights were frozen or finetuned
- decoder design
- expected benefits
- Model definitions are in `src/models.py`

### 3.5 Model Training
Describe:
- how to initialize the model
- training procedure
- optimizer
- learning rate
- batch size
- number of epochs
- early stopping / checkpoint saving
- hardware used
- Training code is in `[insert notebook/script]`
- Saved weights are stored in `checkpoints/`

### 3.6 Hyperparameter Search / Tuning
Describe the experiments performed.

Possible items:
- learning rate
- batch size
- latent dimension / bottleneck size
- augmentation settings
- threshold selection
- pretrained vs non-pretrained encoder

You can present this as a short table.

### 3.7 Loss Functions
Explain the loss function(s) used.

**Example:**
- reconstruction loss (MSE / L1 / SSIM-based loss if applicable)
- any auxiliary losses

### 3.8 Evaluation Metrics
Explain the metrics used to evaluate performance.

Possible metrics:
- pixel-level AUROC
- IoU
- F1-score
- precision / recall
- image-level AUROC
- reconstruction loss trends

State clearly how thresholds were chosen for binary segmentation.

---

## Results
▪ Visualization of model performance
• Accuracy and loss curves
• Performance on validation set images


### 4.1 Training Curves
Include:
- training loss curve
- validation loss curve

Comment on:
- convergence
- overfitting / underfitting
- stability of training

### 4.2 Qualitative Results
Show example outputs on validation / test images.

Include:
- original image
- ground truth mask
- reconstruction
- anomaly map
- predicted binary mask

### 4.3 Quantitative Results
Present final metrics in a table.

**Example table:**

| Model | Pixel AUROC | IoU | F1-score | Notes |
|------|-------------|-----|----------|------|
| Baseline Autoencoder | [x] | [x] | [x] | No augmentation |
| Pretrained Encoder | [x] | [x] | [x] | ResNet18 encoder |
| Best Tuned Model | [x] | [x] | [x] | With augmentation |

### 4.4 Performance on Validation Set Images
Briefly summarize how the model behaved on validation images and whether validation loss aligned with final anomaly detection performance.

---

## Discussion
▪ Examples of model malfunctioning (if any)
▪ Comparison of model performance against state-of-the-art

### 5.1 What Worked Well
Describe the strongest parts of your approach.

### 5.2 Failure Cases / Malfunctioning Examples
Show examples where the model failed.

Discuss issues such as:
- false positives
- missed defects
- blurry reconstructions
- poor localization
- sensitivity to lighting / texture / cable structure

### 5.3 Comparison Against State-of-the-Art
Briefly compare your results with established methods.

Possible methods:
- PatchCore
- PaDiM
- FastFlow
- Anomalib baselines

Discuss:
- accuracy gap
- computational cost
- implementation complexity
- fairness of comparison

### 5.4 Limitations
State limitations of the current project.

Possible examples:
- trained on only one category
- limited hyperparameter search
- simple decoder design
- threshold selection limitations
- no cross-category generalization

### 5.5 Future Work
Possible directions:
- better anomaly scoring
- stronger decoder / skip connections
- better thresholding strategy
- comparison on more MVTec categories
- more advanced anomaly detection methods

---

## How to Run the Code

### Requirements
- Python 3 (tested on v3.13.4)
- OpenCV (tested on 4.13.0)
- PyTorch (tested on 2.10.0)
- Matplotlib (tested on v3.10.8)
- Numpy (tested on v2.4.1)

**Installing dependencies:** `pip install -r requirements.txt`

### Dataset Setup
Explain where to place the dataset.

### Running the Project

Run the notebooks in the following order
1. 01_data_exploration.ipynb
2. 02_baseline_autoencoder.ipynb
3. 03_pretrained_encoder.ipynb
4. 04_augmentation_and_tuning.ipynb
5. 05_final_report.ipynb

### Training from Scratch

Explain how to train the model from scratch.

### Loading a Saved Model

Explain how to load saved weights to reproduce the reported performance curves and metrics.

- checkpoint filename
- where it is stored
- which notebook/script loads it

## Contributions of each group member
- Ang Li En Eldrick
    - a
    - b
- Malvin Ken Sudirgo
    - a
    - b
- Toh Jia Le
    - Project Proposal
    - b

## References

- Paul Bergmann, Michael Fauser, David Sattlegger, and Carsten Steger, "A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection", IEEE Conference on Computer Vision and Pattern Recognition, 2019